# Nemotron Colab GPU Test

This notebook is set up to run end to end in Google Colab with a GPU runtime.

Before running:
- In Colab, open `Runtime -> Change runtime type`
- Set `Hardware accelerator` to `GPU`
- Run the notebook from top to bottom


In [ ]:
!pip install -q -U "transformers==5.2.0" onnx onnxscript accelerate

In [ ]:
import warnings
import torch
from torch.jit import TracerWarning

warnings.filterwarnings("ignore", message="`use_return_dict` is deprecated! Use `return_dict` instead!")
warnings.filterwarnings("ignore", message="`input_embeds` is deprecated and will be removed.*", category=FutureWarning)
warnings.filterwarnings("ignore", message="You are using the legacy TorchScript-based ONNX export.*", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="`cache_position` is deprecated as an arg.*", category=FutureWarning)
warnings.filterwarnings("ignore", category=TracerWarning)

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime not detected. In Colab, enable a GPU runtime before continuing.")

device = "cuda"

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import logging
import sys

import transformers.modeling_attn_mask_utils as attn_mask_utils
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask

model_name = "nvidia/llama-nemotron-embed-vl-1b-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype="auto",
    attn_implementation="eager"
).eval().to(device)
model.config.use_cache = False
attn_mask_utils.logger.setLevel(logging.ERROR)

# ONNX export is more reliable if we avoid the native bidirectional mask path.
remote_module = sys.modules[model.__class__.__module__]
remote_module._prepare_4d_attention_mask = _prepare_4d_attention_mask
if hasattr(remote_module, "_HAS_NATIVE_BIDIRECTIONAL_MASK"):
    remote_module._HAS_NATIVE_BIDIRECTIONAL_MASK = False

# Patch any bound bidirectional-mask methods directly so their globals resolve correctly.
patched_mask_helpers = 0
for submodule in model.modules():
    if hasattr(submodule, "_create_bidirectional_mask"):
        mask_globals = submodule._create_bidirectional_mask.__func__.__globals__
        mask_globals["_prepare_4d_attention_mask"] = _prepare_4d_attention_mask
        mask_globals["_HAS_NATIVE_BIDIRECTIONAL_MASK"] = False
        patched_mask_helpers += 1

print("Model loaded on:", next(model.parameters()).device)
print("Attention implementation:", getattr(model.config, "_attn_implementation", None))
print("Patched bidirectional mask helpers:", patched_mask_helpers)


In [ ]:
def pool_embeddings(last_hidden_states, attention_mask, pool_type):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)

    if pool_type == "avg":
        return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
    if pool_type == "weighted_avg":
        return last_hidden.sum(dim=1)
    if pool_type == "cls":
        return last_hidden[:, 0]
    if pool_type == "last":
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden.shape[0]
        return last_hidden[
            torch.arange(batch_size, device=last_hidden.device), sequence_lengths
        ]
    if pool_type == "cls_last":
        return last_hidden[:, 0]
    if pool_type == "colbert":
        return last_hidden
    raise ValueError(f"Unsupported pooling type: {pool_type}")

pool_type = getattr(model.config, "pooling", "avg")
print("Pooling type:", pool_type)


In [ ]:
text = "test input"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True, return_dict=True, use_cache=False)
    embeddings = pool_embeddings(outputs.hidden_states[-1], inputs["attention_mask"], pool_type)

print("Input IDs shape:", tuple(inputs["input_ids"].shape))
print("Attention mask shape:", tuple(inputs["attention_mask"].shape))
print("Embedding shape:", tuple(embeddings.shape))


In [ ]:
class TextEmbeddingWrapper(torch.nn.Module):
    def __init__(self, base_model, pool_type):
        super().__init__()
        self.base_model = base_model
        self.pool_type = pool_type

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
        return pool_embeddings(outputs.hidden_states[-1], attention_mask, self.pool_type)

embedding_model = TextEmbeddingWrapper(model, pool_type).eval()

from pathlib import Path
import shutil

export_dir = Path("onnx_export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / "model.onnx"

torch.onnx.export(
    embedding_model,
    (inputs["input_ids"], inputs["attention_mask"]),
    str(export_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["embeddings"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "embeddings": {0: "batch"}
    },
    opset_version=17,
    dynamo=False
)

print(f"Exported ONNX model to {export_path.resolve()}")
print("Export directory contents:", sorted(p.name for p in export_dir.iterdir()))


In [ ]:
import os

print("Export directory:", export_dir.resolve())
print("model.onnx exists:", export_path.exists())
print("model.onnx size (MB):", round(export_path.stat().st_size / (1024 * 1024), 2))
total_export_size_mb = round(sum(p.stat().st_size for p in export_dir.iterdir() if p.is_file()) / (1024 * 1024), 2)
print("Total export size (MB):", total_export_size_mb)
print("Exported files:", sorted(p.name for p in export_dir.iterdir()))


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import shutil
from pathlib import Path

drive_export_dir = Path("/content/drive/MyDrive/nemotron_exports")
drive_export_dir.mkdir(parents=True, exist_ok=True)
drive_export_path = drive_export_dir / export_dir.name
if drive_export_path.exists():
    shutil.rmtree(drive_export_path)
shutil.copytree(export_dir, drive_export_path)

print(f"Copied ONNX export bundle to {drive_export_path}")
print("Drive bundle contents:", sorted(p.name for p in drive_export_path.iterdir()))


In [ ]:
import shutil
from google.colab import files
from pathlib import Path

# Optional: zip the full ONNX bundle, including external data files, then download it.
archive_base = Path("onnx_export_bundle")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=export_dir)
print(f"Created archive: {archive_path}")
files.download(archive_path)
